# Agent integration tests

Runs each step of `agents/agent_integration.py` on both datasets and checks the results.
It uses the saved semantic run (offline mode), so no API key is needed.
Every test uses `assert`, so the notebook stops at the first failure.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

def find_repo_root():
    # Works whether the notebook starts in notebooks/, the repo, or the folder above it
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in (folder, folder / "D37-Semantic-Data-Lake"):
            if (candidate / "agents" / "agent_integration.py").exists():
                return candidate
    raise FileNotFoundError("Could not find the D37-Semantic-Data-Lake folder")


# agent_integration uses paths relative to the repository root
ROOT = find_repo_root()
os.chdir(ROOT)
print("Repository root:", ROOT)
sys.path.insert(0, str(ROOT))

from agents import agent_integration as ai

pd.set_option("display.max_colwidth", 100)

Repository root: /Users/victorchen/Desktop/dsc180a/D37-Semantic-Data-Lake


## 1. Run the metadata agent

In [2]:
metadata = ai.run_metadata_agent(ai.DATASETS)

pd.DataFrame({
    key: {"rows": p["row_count"], "columns": p["column_count"], "potential_keys": p["potential_keys"]}
    for key, p in metadata.items()
}).T

Metadata exported to outputs/agent_integration/metadata/heart_metadata.json
Metadata exported to outputs/agent_integration/metadata/cardio_metadata.json


,rows,columns,potential_keys
heart,297,16,[patient_id]
cardio,1000,15,[patientid]


## 2. Load the semantic agent's output

Reuses the latest saved run in `outputs/semantic_agent_profiles/`.

In [3]:
run, semantic = ai.run_semantic_agent(ai.DATASETS, model="gpt-oss", offline=True)
print(f"Using semantic run test_{run:03d}")

for key in ai.DATASETS:
    assert set(semantic[key]["columns"]) == set(metadata[key]["columns"]), f"{key}: columns differ"
print("PASS: semantic profiles cover every column in both datasets")

Using semantic run test_001
PASS: semantic profiles cover every column in both datasets


## 3. Test: metadata statistics are correct

Recalculates every count, min, max, mean and median with pandas and compares them with the metadata agent's pandas and SQL results.

In [4]:
for key, spec in ai.DATASETS.items():
    result = ai.check_metadata(spec, metadata[key], ai.EDA_FACTS[key])
    assert result["passed"] == result["checks"], f"{key} failed: {result['failed']}"
    assert result["primary_key_found"], f"{key}: primary key not found"
    print(f"PASS {key}: {result['passed']}/{result['checks']} checks, primary key found")

PASS heart: 196/196 checks, primary key found
PASS cardio: 184/184 checks, primary key found


## 4. Raw data: does the metadata agent catch the problems found in EDA?

Runs the metadata agent on the original CSVs.
- **detected**: the agent reports the problem.
- **visible**: the problem shows up only indirectly (e.g. `min = 0`).
- **missed**: no trace of it.

This is a finding about the agent, not a pass/fail test.

In [5]:
raw_issues = pd.DataFrame([
    {"dataset": key, **issue}
    for key, spec in ai.DATASETS.items()
    for issue in ai.check_raw_issues(spec, ai.EDA_FACTS[key])
])
raw_issues

,dataset,issue,outcome
0,heart,4 missing values in ca,detected
1,heart,2 missing values in thal,detected
2,heart,restecg = 1 has only 4 patients,missed
3,heart,thal = 6 has only 18 patients,missed
4,cardio,53 cholesterol readings of 0 are really missing,visible
5,cardio,slope = 0 (180 rows) is undocumented,visible
6,cardio,all missing cholesterol is in target = 1 patients,missed


## 5. Semantic agent vs data dictionaries

One row per column: is the meaning right, do the unit and value codes match the data dictionary, and did the agent flag the column as uncertain?

In [6]:
semantic_results = pd.DataFrame([
    {"dataset": key, **row}
    for key, spec in ai.DATASETS.items()
    for row in ai.check_semantic(key, spec, semantic[key])
])
semantic_results.drop(columns="note")

,dataset,column,meaning,unit,codes,flagged,error
0,heart,patient_id,correct,NaN,NaN,False,False
1,heart,age,correct,match,NaN,False,False
2,heart,sex,correct,NaN,match,False,False
3,heart,cp,correct,NaN,match,False,False
4,heart,trestbps,correct,match,NaN,False,False
5,heart,chol,correct,match,NaN,False,False
6,heart,fbs,correct,NaN,match,False,False
7,heart,restecg,correct,NaN,match,False,False
8,heart,thalach,correct,match,NaN,False,False
9,heart,exang,correct,NaN,match,False,False


In [7]:
# Summary per dataset
summary = pd.DataFrame({
    key: {
        "columns": len(group),
        "meaning correct": int((group["meaning"] == "correct").sum()),
        "columns with errors": int(group["error"].sum()),
        "errors flagged by agent": int((group["error"] & group["flagged"]).sum()),
    }
    for key, group in semantic_results.groupby("dataset", sort=False)
}).T
summary

,columns,meaning correct,columns with errors,errors flagged by agent
heart,16,14,2,2
cardio,15,13,2,1


In [8]:
# Columns the agent got wrong
semantic_results[semantic_results["error"]][["dataset", "column", "meaning", "codes", "flagged", "note"]]

,dataset,column,meaning,codes,flagged,note
13,heart,thal,partially_correct,match,True,calls it thalassemia; human profile says thallium stress test (codes right)
14,heart,num,partially_correct,partial,True,"first reading (vessel count 0-3) is wrong; second (0 = no disease, 1-4 = disease) is right"
27,cardio,slope,partially_correct,mismatch,True,"labels in wrong codes: implies 0 = upsloping; human has 0 = undocumented, 1-3 = up/flat/down"
28,cardio,noofmajorvessels,partially_correct,NaN,False,"adds 'with significant narrowing', which is unsupported and not flagged"


## 6. Relationships across datasets

Compares the metadata agent's same-name matches with the 14 column pairs in the human crosswalk.

In [9]:
relationships = ai.run_relationships(ai.DATASETS)
relationship_results = pd.DataFrame(ai.check_relationships(relationships))

for row in relationship_results.itertuples():
    assert row.correct == row.found, "the agent proposed a pair that is not in the crosswalk"
print("PASS: every pair the agent found is correct")
relationship_results

PASS: every pair the agent found is correct


,pair,expected,found,correct,missed
0,heart vs cardio,14,3,3,"[ca/noofmajorvessels, chol/serumcholestrol, cp/chestpain, disease/target, exang/exerciseangia, f..."


## 7. Unit tests for the comparison helpers

These check the text-parsing functions on known examples, including a bug found during development (`absence of heart disease` vs `no disease`).

In [10]:
# parse_codes: explicit mappings, ranges, LLM spacing, positional lists, no codes
assert ai.parse_codes("0 = normal, 1 = ST-T wave abnormality, 2 = LVH") == {0: "normal", 1: "ST-T wave abnormality", 2: "LVH"}
assert ai.parse_codes("0 = no disease; 1\u20134 = disease present") == {0: "no disease", 1: "disease present", 2: "disease present", 3: "disease present", 4: "disease present"}
assert ai.parse_codes("Sex (1\u202f=\u202fmale, 0\u202f=\u202ffemale).") == {1: "male", 0: "female"}
assert ai.parse_codes("Slope coded as 0\u20113 (upsloping, flat, downsloping, unknown).") == {0: "upsloping", 1: "flat", 2: "downsloping", 3: "unknown"}
assert ai.parse_codes("Number of major vessels (0-3)") == {}
print("PASS: parse_codes")

# labels_match
assert ai.labels_match("absence of heart disease", "no disease")
assert ai.labels_match("probable/definite left ventricular hypertrophy (Estes' criteria)", "left ventricular hypertrophy")
assert not ai.labels_match("typical angina", "atypical angina")
assert not ai.labels_match("upsloping", "flat")
print("PASS: labels_match")

# compare_unit
assert ai.compare_unit("mm Hg", "mmHg") == "match"
assert ai.compare_unit("beats/min (71\u2013202)", "beats per minute") == "match"
assert ai.compare_unit("0.0\u20136.2 (measurement unit not specified)", "mm (hypothesized)") == "hedged_guess"
assert ai.compare_unit("0.0\u20136.2 (measurement unit not specified)", "mm") == "unsupported"
assert ai.compare_unit("mg/dL", None) == "missing"
assert ai.compare_unit("1 = true, 0 = false", None) is None
print("PASS: compare_unit")

# compare_codes
assert ai.compare_codes("0 = female, 1 = male", "Sex (1 = male, 0 = female)") == "match"
assert ai.compare_codes("0 = absence of heart disease, 1 = presence of heart disease",
                        "Presence of heart disease (1 = disease, 0 = no disease).") == "match"
assert ai.compare_codes("0 = undocumented category (180 records), 1 = upsloping, 2 = flat, 3 = downsloping",
                        "Slope of the peak exercise ST segment, coded as 0\u20113 (upsloping, flat, downsloping, unknown).") == "mismatch"
assert ai.compare_codes("0\u20133", "anything") is None
print("PASS: compare_codes")

PASS: parse_codes
PASS: labels_match
PASS: compare_unit
PASS: compare_codes


## 8. Test: results are reproducible

Running the evaluation twice must give identical results, and the summary must match the saved `test_001` output.

In [11]:
def evaluate():
    return {key: ai.evaluate_dataset(key, spec, metadata[key], semantic[key]) for key, spec in ai.DATASETS.items()}

current = evaluate()
assert current == evaluate()
print("PASS: two evaluation runs give identical results")

saved = json.loads(Path("outputs/agent_integration/test_001_evaluation_results.json").read_text())
for key in ai.DATASETS:
    assert current[key]["summary"] == saved["datasets"][key]["summary"], f"{key}: differs from test_001"
print("PASS: summary matches saved test_001 results")

PASS: two evaluation runs give identical results
PASS: summary matches saved test_001 results
